# Soil Moisture Sensor Calibration

This notebook calibrates a soil moisture sensor by comparing raw sensor readings (0–1023) against gravimetric soil moisture content measured from physical soil samples.

**Formula:**

```
Soil Moisture % = ((Wwet - Wdry) / Wdry) × 100
```

## Step 1 – Load calibration data from CSV

We load the soil sample measurements recorded in March 2026 from `soil_moisture_samples.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the CSV dataset
df = pd.read_csv('soil_moisture_samples.csv')

# Calculate gravimetric soil moisture % for each sample
df['moisture_pct'] = ((df['weight_wet_g'] - df['weight_dry_g']) / df['weight_dry_g']) * 100

print('Calibration data loaded:')
print(df[['sample', 'date', 'sensor_reading', 'weight_wet_g', 'weight_dry_g', 'moisture_pct']].to_string(index=False))

## Step 2 – Plot the calibration graph

We plot sensor reading (x-axis) against soil moisture % (y-axis), then fit a best-fit line through the points.

In [ ]:
# Fit a linear best-fit line
coeffs = np.polyfit(df['sensor_reading'], df['moisture_pct'], 1)
best_fit_line = np.poly1d(coeffs)

x_line = np.linspace(0, 1023, 300)
y_line = best_fit_line(x_line)

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(df['sensor_reading'], df['moisture_pct'],
           color='steelblue', s=80, zorder=5, label='Calibration samples')

for _, row in df.iterrows():
    ax.annotate(f"Sample {int(row['sample'])}\n({int(row['sensor_reading'])}, {row['moisture_pct']:.1f}%)",
                xy=(row['sensor_reading'], row['moisture_pct']),
                xytext=(10, 5), textcoords='offset points', fontsize=8, color='dimgray')

ax.plot(x_line, y_line, color='tomato', linewidth=1.8,
        label=f'Best-fit line: y = {coeffs[0]:.4f}x + {coeffs[1]:.2f}')

ax.set_xlabel('Soil moisture sensor reading (0-1023)', fontsize=11)
ax.set_ylabel('Gravimetric soil moisture (%)', fontsize=11)
ax.set_title('Soil Moisture Sensor Calibration - March 2026', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
ax.set_xlim(0, 1023)
ax.set_ylim(0, max(df['moisture_pct']) * 1.3)

plt.tight_layout()
plt.savefig('calibration_graph.png', dpi=150)
plt.show()
print('Graph saved as calibration_graph.png')

## Step 3 – Convert a live sensor reading to soil moisture %

Using the best-fit line, we can now convert any raw sensor reading into a gravimetric soil moisture percentage.

In [ ]:
# -----------------------------------------------------------
# Enter your live sensor reading here
# -----------------------------------------------------------
live_sensor_reading = 520

estimated_moisture = best_fit_line(live_sensor_reading)

print(f'Live sensor reading     : {live_sensor_reading}')
print(f'Estimated soil moisture : {estimated_moisture:.2f}%')

fig2, ax2 = plt.subplots(figsize=(9, 5))

ax2.scatter(df['sensor_reading'], df['moisture_pct'],
            color='steelblue', s=80, zorder=5, label='Calibration samples')
ax2.plot(x_line, y_line, color='tomato', linewidth=1.8, label='Best-fit line')

ax2.axvline(x=live_sensor_reading, color='green', linestyle='--', linewidth=1.2, alpha=0.7)
ax2.axhline(y=estimated_moisture, color='green', linestyle='--', linewidth=1.2, alpha=0.7)
ax2.scatter([live_sensor_reading], [estimated_moisture],
            color='green', s=120, zorder=6,
            label=f'Live reading: {live_sensor_reading} -> {estimated_moisture:.1f}%')

ax2.set_xlabel('Soil moisture sensor reading (0-1023)', fontsize=11)
ax2.set_ylabel('Gravimetric soil moisture (%)', fontsize=11)
ax2.set_title('Reading converted using calibration line', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, linestyle='--', alpha=0.4)
ax2.set_xlim(0, 1023)
ax2.set_ylim(0, max(df['moisture_pct']) * 1.3)

plt.tight_layout()
plt.savefig('live_reading_graph.png', dpi=150)
plt.show()

## Summary

| Step | Result |
|------|--------|
| Calibration samples collected | 5 samples (March 2026) |
| Sensor reading range tested | 110 – 823 |
| Best-fit line equation | printed above |
| Live sensor reading used | 520 |
| Estimated soil moisture | calculated above |

**Conclusion:** By weighing wet and dry soil samples and matching them to sensor readings, we built a calibration curve. Any future raw sensor value can now be converted to a gravimetric soil moisture percentage using the best-fit line.